In [0]:
bronze = "/Volumes/projeto_databricks/logistics/bikestore/bronze/"
silver = "/Volumes/projeto_databricks/logistics/bikestore/silver/"
gold = "/Volumes/projeto_databricks/logistics/bikestore/gold/"
resource = "/Volumes/projeto_databricks/logistics/bikestore/resource/"
origem = "/Volumes/projeto_databricks/logistics/bikestore/resource/origem/"
destino = "/Volumes/projeto_databricks/logistics/bikestore/resource/destino/"

In [0]:
display(dbutils.fs.ls(f'{bronze}'))

In [0]:
display(dbutils.fs.ls(f'{bronze}/brands/'))

In [0]:
%sql
--brands tabela física
--CREATE TABLE if not EXISTS bikestore.logistics.bronze_brands
--LOCATION '/Volumes/projeto_databricks/logistics/bikestore/bronze/brands/'


In [0]:
'''
ETL com tabela temporária, geralmente nao se cria tabelas fisicas para a camada bronze, 
ela é ultilizada para o refinamento e dai sim criar tabelas fisicas em camada silver 
'''

In [0]:
# criando tabela temporária - para usar na proxima camada
(spark.read.format('delta')
  .load(f'{bronze}/customers/')
  .createOrReplaceTempView('tmp_customers')

) 


In [0]:
%sql
select  * from tmp_customers

In [0]:
# criar varias tabelas temporárias de forma prática
bronze_map = {
    #nome da tabela temporária: caminho da pasta que estão os arquivos
    "tmp_bronze_brands":      f"{bronze}/brands/",
    "tmp_bronze_categories":  f"{bronze}/categories/",
    "tmp_bronze_customers":   f"{bronze}/customers/",
    "tmp_bronze_order_items": f"{bronze}/order_items/",
    "tmp_bronze_orders":      f"{bronze}/orders/",
    "tmp_bronze_products":    f"{bronze}/products/",
    "tmp_bronze_staffs":      f"{bronze}/staffs/",
    "tmp_bronze_stocks":      f"{bronze}/stocks/",
    "tmp_bronze_stores":      f"{bronze}/stores/",
}
for view_name, path in bronze_map.items():
    (spark.read.format('delta')
        .load(path)   #vamos carregar todos os caminhos que estão no dicionário
        .createOrReplaceTempView(view_name))
 


In [0]:
%sql
--verificando se deu certo a criação de todas as tabelas temporárias
show tables in projeto_databricks.logistics

In [0]:
%sql
select * from tmp_bronze_stores

In [0]:
describe tmp_bronze_brands

Criando uma camada Silver de teste


In [0]:
%sql
select * from tmp_bronze_products limit 10

In [0]:
%sql
select * from tmp_bronze_categories

In [0]:
%sql
describe tmp_bronze_categories

In [0]:
%sql
select
   p.product_id
  ,p.product_name
  ,p.brand_id
  --,p.category_id
  --,c.category_id as category_id_categoriat
  ,c.category_name
  ,p.model_year
  ,p.list_price
from        tmp_bronze_products as p
left join tmp_bronze_categories as c on p.category_id = c.category_id



In [0]:
df_product_silver=spark.sql("""
select
   p.product_id
  ,p.product_name
  ,p.brand_id
  --,p.category_id
  --,c.category_id as category_id_categoriat
  ,c.category_name
  ,p.model_year
  ,p.list_price
from        tmp_bronze_products as p
left join tmp_bronze_categories as c on p.category_id = c.category_id

                            """) 

In [0]:
display (df_product_silver)

In [0]:
#salvar em parquet como delta  tabela de teste SILVER
df_product_silver.write\
    .mode('overwrite')\
    .format('delta')\
    .option('mergeSchema','true')\
    .save(f'{silver_path}/product_teste')



In [0]:
%sql
/*Criar tabela fisica de teste este comando precisa rodar só 1x pois quando arquivos forem atualizados a tabela irá refletir o resultado  (irá pegar o log mais recente e trazer dados do arquivo(os) atualizados)*/
CREATE TABLE IF NOT EXISTS bikestore.logistics.products_teste
LOCATION 'abfss://uc-ext-azure@externalazure.dfs.core.windows.net/bikestore/silver/product_teste';



In [0]:
%sql
select * from bikestore.logistics.products_teste